# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# NB_08 — Embeddings con `ai.embed`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo:** `text-embedding-ada-002`  
**Endpoint:** integrado de Fabric

**Objetivo:** transformar textos en vectores, validar dimensiones, preparar una tabla Delta y auditar tokens y capacidad. Los embeddings permiten comparar significado, no son respuestas de texto ni identificadores únicos.

## 1. Requisitos y consideraciones

- Workspace con capacidad compatible y AI Functions habilitadas por el administrador.
- Notebook PySpark con SynapseML disponible; en Runtime 2.0 consultar la preparación temporal descrita en la [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).
- Usar una sesión limpia con endpoint integrado; no configurar claves externas. Lakehouse adjunto solo para persistencia opcional.
- Datos ficticios; no enviar secretos ni datos personales. Los vectores derivados también requieren controles de acceso.

## 2. Configurar el modelo y las librerías

El modelo de embeddings tiene configuración independiente del modelo generativo.

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "text-embedding-ada-002"

aifunc.default_conf.set_embedding_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")
from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array
EXPECTED_DIM = 1536

StatementMeta(, e90e7f7a-f590-44da-95d5-d215fb616ec0, 3, Finished, Available, Finished, False)

Modelo configurado: text-embedding-ada-002


## 3. Referencia del método publicado por Microsoft

[Microsoft Learn — Use ai.embed with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/embed)

### Sintaxis documentada

```python
df.ai.embed(input_col="col1", output_col="embed")
```

| Parámetro | Tipo | Requerido | Comportamiento |
|---|---|---|---|
| input_col | string | Sí | Columna de texto existente |
| output_col | string | No | Nombre de salida; se genera uno si se omite |
| error_col | string | No | Nombre de columna de errores; se genera uno si se omite |

**Retorno:** DataFrame con vectores DenseVector. El tamaño depende del modelo. Para el modelo Ada esperamos 1536 componentes; se valida empíricamente abajo. La [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration) explica también `deploymentName` por llamada. No se necesita un prompt.

## 4. Crear datos ficticios

Pares de tickets con temas cercanos y otros diferentes para explorar representaciones semánticas.

In [2]:
data = [(1, "No puedo acceder al portal corporativo"),
        (2, "El inicio de sesión en el portal falla"),
        (3, "La carga nocturna de ventas no terminó"),
        (4, "El pipeline de ventas falló durante la madrugada"),
        (5, "Solicito una silla ergonómica para la oficina"),
        (6, "Necesito restablecer mi contraseña")]
df = spark.createDataFrame(data, ["id_ticket", "descripcion"])
display(df)

StatementMeta(, e90e7f7a-f590-44da-95d5-d215fb616ec0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b803041d-68d9-471e-9ee4-5e62cb809a58)

## 5. Generar embeddings

Materializamos el resultado de este conjunto pequeño una vez. La caché reduce reevaluaciones, pero su pérdida puede provocar nuevas llamadas.

In [3]:
run_id = str(uuid.uuid4())
execution_timestamp_utc = datetime.now(timezone.utc).isoformat()
resultado = df.ai.embed(input_col="descripcion", output_col="embedding", error_col="error_embedding").persist(StorageLevel.MEMORY_AND_DISK)
resultado.collect()
resultado.printSchema()

StatementMeta(, e90e7f7a-f590-44da-95d5-d215fb616ec0, 5, Finished, Available, Finished, False)

root
 |-- id_ticket: long (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- error_embedding: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = false)
 |    |    |    |-- minor: integer (nullable = false)
 |    |    |-- statusCode: integer (nullable = false)
 |    |    |-- reasonPhrase: string (nullable = true)
 |-- embedding: vector (nullable = true)



## 6. Validar resultados y dimensionalidad

Convertimos los vectores ML a arrays de doubles para interoperabilidad con Delta. Mostramos solo cinco componentes y el tamaño, no el vector completo. Los errores se conservan.

In [4]:
# Una UDF tolerante a nulos evita invocar vector_to_array sobre vectores ausentes.
from pyspark.sql.types import ArrayType
@F.udf(ArrayType(DoubleType()))
def to_array_safe(v):
    return None if v is None else [float(a) for a in v]

vectores = resultado.withColumn("embedding_array", to_array_safe("embedding"))
vectores = vectores.withColumn("dimension", F.when(F.col("embedding_array").isNotNull(), F.size("embedding_array")))
vectores = vectores.withColumn("validation_status", F.when(F.col("error_embedding").isNotNull(), "ERROR").when(F.col("dimension").isNull(), "MISSING").when(F.col("dimension") != EXPECTED_DIM, "INVALID_DIM").otherwise("OK"))
display(vectores.select("id_ticket", "descripcion", "dimension", F.slice("embedding_array", 1, 5).alias("preview"), "validation_status", "error_embedding"))
display(vectores.groupBy("validation_status", "dimension").count())

StatementMeta(, e90e7f7a-f590-44da-95d5-d215fb616ec0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 871c22fd-deb8-462d-afe2-20f71497947e)

SynapseWidget(Synapse.DataFrame, 38f2b4a3-f52f-4997-98d4-9d55582933a2)

## 7. Consultar estadísticas con `ai.stats`

Las estadísticas se consultan desde el DataFrame resultante de la operación:

```python
stats_df = resultado.ai.stats
display(stats_df)
```

La documentación incluye métricas de ejecución y consumo de tokens, entre ellas `num_successful`, `num_exceptions`, `num_unevaluated`, `input_tokens`, `cached_tokens`, `output_tokens` y `reasoning_tokens`.

La disponibilidad de cada métrica depende de la respuesta del servicio. Por eso, el código consulta los campos de forma defensiva.

In [ ]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

if len(stats_rows) != 1:
    raise RuntimeError("Se esperaba una fila de estadísticas; revisar stats_df antes de estimar.")
stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI EMBED =====")
for key, value in stats.items():
    print(f"{key}: {value}")

## 8. Estimar el consumo de capacidad

[Facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing), consultada el 25-11-2026: Ada, 3.36 CU-s por 1000 tokens de entrada; contexto 8192 tokens.

$$CU_s = I \times 3.36 / 1000$$

Se estima solamente la operación de modelo integrada, no Spark. No aplicamos tarifas generativas. Si falta la métrica de entrada o el modelo/endpoint no coincide, no calculamos un costo ficticio. Contrastar con Capacity Metrics > AI Functions.

In [ ]:
def optional_int(key):
    value = stats.get(key)
    return None if value is None else int(value)
input_tokens = optional_int("input_tokens")
cached_tokens = optional_int("cached_tokens")
output_tokens = optional_int("output_tokens")
reasoning_tokens = optional_int("reasoning_tokens")
RATE_INPUT = 3.36
reported_model = str(stats.get("model") or "")
client_type = str(stats.get("client_type") or "")
if input_tokens is not None and input_tokens >= 0 and reported_model == MODEL_NAME and client_type == "fabric_llm_endpoint":
    estimated_cu_seconds = input_tokens * RATE_INPUT / 1000
else:
    estimated_cu_seconds = None
    print("Estimación no disponible: revisar modelo, endpoint o tokens reportados.")
print(f"Tokens de entrada: {input_tokens}")
print(f"Tokens en caché: {cached_tokens}")
print(f"Tokens de salida: {output_tokens}")
print(f"Tokens de razonamiento: {reasoning_tokens}")
print(f"CU-segundos estimados: {estimated_cu_seconds}")

## 9. Construir un registro de auditoría

Conservamos las mismas columnas de la serie. Los valores no reportados permanecen nulos; no equivalen a cero. La auditoría no contiene textos ni vectores.

In [ ]:
def metric_int(name):
    value = stats.get(name)
    return int(value) if value is not None else None

num_input_rows = df.count()
num_successful = metric_int("num_successful")
num_exceptions = metric_int("num_exceptions")
num_unevaluated = metric_int("num_unevaluated")
num_harmful = metric_int("num_harmful")

if any(v is None for v in (num_successful, num_exceptions, num_harmful, num_unevaluated)):
    execution_status = "UNKNOWN"
elif num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == num_input_rows:
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_08_AI_EMBED",
    "ai.embed",
    MODEL_NAME,
    stats.get("model"),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    estimated_cu_seconds,
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

## 10. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 11. Ejemplo adicional: comparar vectores sin llamar nuevamente al modelo

Usamos similitud coseno entre el ticket 1 y los restantes. No se establece un umbral universal. Este ejemplo local de Spark no incurre en nuevos tokens si el resultado sigue en caché; sí usa cómputo Spark. El siguiente notebook explicará `ai.similarity`.

In [ ]:
valid = vectores.filter(F.col("validation_status") == "OK")
a = valid.filter(F.col("id_ticket") == 1).select(F.col("embedding_array").alias("a"))
b = valid.filter(F.col("id_ticket") != 1).select("id_ticket", "descripcion", F.col("embedding_array").alias("b"))
pairs = b.crossJoin(a)
cos = pairs.withColumn("dot", F.expr("aggregate(zip_with(a,b,(x,y) -> x*y), 0D, (s,x) -> s+x)"))
cos = cos.withColumn("norm_a", F.expr("sqrt(aggregate(a,0D,(s,x) -> s+x*x))")).withColumn("norm_b", F.expr("sqrt(aggregate(b,0D,(s,x) -> s+x*x))"))
cos = cos.withColumn("cosine_similarity", F.when((F.col("norm_a") > 0) & (F.col("norm_b") > 0), F.col("dot")/(F.col("norm_a")*F.col("norm_b"))))
display(cos.select("id_ticket", "descripcion", "cosine_similarity").orderBy(F.desc("cosine_similarity")))

## 12. Persistir vectores en Delta (opcional)

El array numérico evita depender del tipo VectorUDT en consumidores posteriores. Guardar modelo, dimensión y run_id permite identificar el espacio vectorial. No mezclar vectores de modelos distintos. La tabla es independiente de la auditoría; append puede duplicar registros al repetir la escritura.

In [ ]:
RUN_SAVE_VECTORS = False
if RUN_SAVE_VECTORS:
    (valid.select("id_ticket", "descripcion", "embedding_array", "dimension")
     .withColumn("embedding_model", F.lit(MODEL_NAME))
     .withColumn("run_id", F.lit(run_id))
     .write.format("delta").mode("append").saveAsTable("dbo.demo_ticket_embeddings"))

## 13. Liberar caché

In [ ]:
resultado.unpersist()

## 14. Conclusiones y referencias

Generamos embeddings, validamos su dimensionalidad, calculamos similitud coseno y preparamos auditoría y persistencia opcional. La tarifa puede cambiar. Los ejemplos se deben ejecutar en Fabric para observar resultados reales.

- [Método ai.embed](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/embed)
- [Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)
- [Descripción general](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)